# Method 1 — DomURLs_BERT Fine-Tuning · RUN 3 (benign top-domain augmentation)

**Why RUN 3:** RUN 2 (combined PhiUSIIL + malicious_phish) fixed cross-dataset
generalization (per-source ROC-AUC ~0.998) BUT scored real `google.com/maps`,
`maybank2u.com.my`, `paypal.com` all ≈ 0.996 phishing. Cause: brand keywords
(paypal/maybank/shopee) appear mostly in PHISHING URLs across both datasets, so the
model learned "brand keyword ⇒ phishing" and flags the genuine brand sites too;
famous short homepage URLs are also out-of-distribution vs crawl-style benign data.

**RUN 3 fix:** add a large set of **real top-domain URLs as benign** (Tranco list —
includes the genuine paypal/maybank/google domains). This teaches "these famous
domains are safe" and breaks the brand-keyword shortcut. Paths are sampled from real
benign URLs so augmented benign URLs are structurally similar to phishing (avoids a
new "has path ⇒ phishing" shortcut). Outputs under `run3_augmented/`; RUN 1 & RUN 2
left intact for the report's generalization case study.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
Checkpoint-resumable per phase (`run3_augmented/`).

**Success = Phase 6 sanity accuracy ≥ 0.9 AND google.com/maps scores LOW.**


In [ ]:
# ============ Phase 0 — Setup ============
%pip -q install transformers datasets accelerate tldextract ucimlrepo onnx onnxruntime "optimum[onnxruntime]" scikit-learn pyarrow tranco

from google.colab import drive
drive.mount('/content/drive')

import os, json, random, time
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

RUN_TAG = 'run3_augmented'
BASE = f'/content/drive/MyDrive/FYP2/method1/{RUN_TAG}'
DATA_DIR = '/content/drive/MyDrive/FYP2/data'
for d in [BASE, f'{BASE}/splits', f'{BASE}/eval', f'{BASE}/artifacts']:
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: enable T4 (Runtime > Change runtime type) before Phase 3.')
print('RUN 2 base:', BASE)


## Phase 1 — Load, harmonise, combine + Tranco benign augmentation
Both fraud datasets are reduced to `url` + binary `label` (**1 = phishing/malicious,
0 = benign/legitimate**) + a `source` tag; only the URL string is used. Then a large
set of **real top-domain URLs (Tranco)** is added as benign — the RUN 3 fix that
breaks the "brand keyword ⇒ phishing" shortcut and covers famous domains.

- PhiUSIIL: majority class = legitimate → label 1 = the minority (phishing).
- malicious_phish: `type != 'benign'` → 1.
- Tranco: top ~150k domains → benign URLs; half get a real benign path (sampled from
  the fraud datasets' benign URLs) so structure matches phishing URLs.


In [ ]:
# ============ Phase 1 — Combine PhiUSIIL + malicious_phish + Tranco benign ============
import pandas as pd
import numpy as np
from urllib.parse import urlsplit

CLEAN_PATH = f'{BASE}/splits/combined_clean.parquet'

if os.path.exists(CLEAN_PATH):
    df = pd.read_parquet(CLEAN_PATH)
    print(f'Phase 1 already completed -- skipping. ({len(df):,} rows)')
    print(df.groupby(['source', 'label']).size())
else:
    frames = []

    # --- PhiUSIIL ---
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=967)
        raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
    except Exception as e:
        print('ucimlrepo failed:', e)
        p = f'{DATA_DIR}/PhiUSIIL_Phishing_URL_Dataset.csv'
        assert os.path.exists(p), f'Upload PhiUSIIL CSV to {p}'
        raw = pd.read_csv(p)
    uc = next(c for c in raw.columns if c.lower() == 'url')
    lc = next(c for c in raw.columns if c.lower() == 'label')
    ph = raw[[uc, lc]].rename(columns={uc: 'url', lc: 'orig'})
    majority = ph['orig'].value_counts().idxmax()
    ph['label'] = (ph['orig'] != majority).astype(int)
    ph['source'] = 'phiusiil'
    frames.append(ph[['url', 'label', 'source']])
    print('PhiUSIIL:', len(ph), ph['label'].value_counts().to_dict())

    # --- malicious_phish ---
    mp_path = f'{DATA_DIR}/malicious_phish.csv'
    assert os.path.exists(mp_path), f'Upload malicious_phish.csv to {mp_path}'
    mp = pd.read_csv(mp_path)
    uc = next(c for c in mp.columns if c.lower() == 'url')
    tc = next(c for c in mp.columns if c.lower() in ('type', 'label'))
    mp = mp[[uc, tc]].rename(columns={uc: 'url', tc: 'type'})
    mp['label'] = (mp['type'].astype(str).str.lower() != 'benign').astype(int)
    mp['source'] = 'malicious_phish'
    frames.append(mp[['url', 'label', 'source']])
    print('malicious_phish:', len(mp), mp['label'].value_counts().to_dict())

    # --- Tranco benign top-domain augmentation (RUN 3 fix) ---
    N_TOP = 150_000
    try:
        from tranco import Tranco
        t = Tranco(cache=True, cache_dir='/content/tranco_cache')
        top = t.list().top(N_TOP)
    except Exception as e:
        print('tranco download failed:', e, '-> Drive CSV fallback')
        fp = f'{DATA_DIR}/tranco_top.csv'
        assert os.path.exists(fp), (
            f'Provide a top-domains CSV at {fp} (rank,domain or one domain per line)')
        tt = pd.read_csv(fp, header=None)
        top = tt.iloc[:, -1].astype(str).tolist()[:N_TOP]

    # Sample real paths from the benign URLs already collected so augmented benign
    # URLs look structurally like phishing (which carry paths) -> no path shortcut.
    benign_urls = pd.concat(frames)
    benign_urls = benign_urls[benign_urls.label == 0]['url']
    def path_of(u):
        try:
            return urlsplit(u if '//' in u else 'http://' + u).path or '/'
        except Exception:
            return '/'
    paths = benign_urls.map(path_of)
    paths = paths[paths.str.len() > 1]
    paths = paths.sample(n=min(60_000, len(paths)), random_state=SEED).tolist()

    rng = np.random.default_rng(SEED)
    aug = []
    for d in top:
        d = str(d).strip().lower()
        if not d or ' ' in d or '.' not in d:
            continue
        if paths and rng.random() < 0.5:
            aug.append(f'https://{d}{paths[int(rng.integers(len(paths)))]}')
        else:
            aug.append(f'https://{d}/')
    tr = pd.DataFrame({'url': aug, 'label': 0, 'source': 'tranco'})
    frames.append(tr)
    print('Tranco benign added:', len(tr))

    # --- combine, clean, dedupe ---
    df = pd.concat(frames, ignore_index=True)
    df['url'] = df['url'].astype(str).str.strip()
    df = df[df['url'].str.len() > 0].dropna(subset=['url'])
    before = len(df)
    conflict = df.groupby('url')['label'].nunique()
    bad = set(conflict[conflict > 1].index)
    df = df[~df['url'].isin(bad)]
    df = df.drop_duplicates(subset='url', keep='first').reset_index(drop=True)
    print(f'Removed {len(bad):,} conflicting + {before - len(bad) - len(df):,} '
          f'duplicate URLs; {len(df):,} remain')
    print('Combined label balance:', df['label'].value_counts().to_dict())
    print(df.groupby(['source', 'label']).size())
    df.to_parquet(CLEAN_PATH)
    print('Phase 1 complete ->', CLEAN_PATH)


## Phase 2 — Domain-level split (keeps source tag)
Grouped by registered domain (tldextract), whole domain groups assigned 70/15/15,
stratified by the domain's majority label. The `source` tag is carried into the
splits so Phase 4 can report per-source test metrics — the direct check that the
model handles both distributions, not just one.

In [ ]:
# ============ Phase 2 — Domain-level 70/15/15 split ============
import tldextract
from sklearn.model_selection import train_test_split

SPLIT_PATHS = {s: f'{BASE}/splits/{s}.parquet' for s in ('train', 'val', 'test')}

if all(os.path.exists(p) for p in SPLIT_PATHS.values()):
    splits = {s: pd.read_parquet(p) for s, p in SPLIT_PATHS.items()}
    print('Phase 2 already completed -- skipping.',
          {s: len(v) for s, v in splits.items()})
else:
    ext = tldextract.TLDExtract(suffix_list_urls=())
    def reg_domain(u):
        try:
            host = u.split('//', 1)[-1].split('/', 1)[0].split('@')[-1].split(':')[0]
            return ext(host).top_domain_under_public_suffix or host.lower()
        except Exception:
            return u.lower()
    df['domain'] = df['url'].map(reg_domain)

    dom = df.groupby('domain')['label'].agg(['mean']).reset_index()
    dom['dom_label'] = (dom['mean'] >= 0.5).astype(int)
    tr_d, rest = train_test_split(dom, test_size=0.30,
                                  stratify=dom['dom_label'], random_state=SEED)
    va_d, te_d = train_test_split(rest, test_size=0.50,
                                  stratify=rest['dom_label'], random_state=SEED)
    sets = {'train': set(tr_d['domain']), 'val': set(va_d['domain']),
            'test': set(te_d['domain'])}
    assert not (sets['train'] & sets['val']) and not (sets['train'] & sets['test'])
    assert not (sets['val'] & sets['test'])
    print('Domain sets disjoint: OK')

    splits = {}
    for s, doms in sets.items():
        part = df[df['domain'].isin(doms)][['url', 'label', 'source']].reset_index(drop=True)
        if s == 'train' and len(part) > 200_000:
            part = part.sample(n=200_000, random_state=SEED).reset_index(drop=True)
            print('Train subsampled to 200,000 URLs (runtime budget).')
        splits[s] = part
        part.to_parquet(SPLIT_PATHS[s])
    for s, part in splits.items():
        by_src = part.groupby('source').size().to_dict()
        print(f'{s:>5}: {len(part):,} | phishing {part.label.mean():.3f} | {by_src}')
    print('Phase 2 complete.')


## Phase 3 — Fine-tuning
`amahdaouy/DomURLs_BERT` + 2-class head (falls back to `bert-base-uncased` with a
loud warning). 3 epochs, lr 2e-5, warmup 10%, fp16, early stopping on val F1.
**CUDA OOM?** set `BATCH = 16`, `GRAD_ACCUM = 2` and re-run.

In [ ]:
# ============ Phase 3 — Train ============
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback, set_seed)
from datasets import Dataset
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score)

MODEL_ID = 'amahdaouy/DomURLs_BERT'
BEST_DIR = f'{BASE}/best_model'
MAX_LEN, BATCH, GRAD_ACCUM = 128, 32, 1

if os.path.exists(f'{BEST_DIR}/config.json'):
    print('Phase 3 already completed -- skipping. Model at', BEST_DIR)
else:
    set_seed(SEED)
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
        used_model = MODEL_ID
    except Exception as e:
        print('!' * 72)
        print('WARNING: could not load', MODEL_ID, '->', repr(e))
        print('FALLING BACK to bert-base-uncased — adjust the report claim!')
        print('!' * 72)
        used_model = 'bert-base-uncased'
        tok = AutoTokenizer.from_pretrained(used_model)
        model = AutoModelForSequenceClassification.from_pretrained(used_model, num_labels=2)

    def tokenize(b): return tok(b['url'], truncation=True, max_length=MAX_LEN)
    ds = {s: Dataset.from_pandas(splits[s][['url', 'label']]).map(
              tokenize, batched=True, remove_columns=['url']) for s in splits}

    def compute_metrics(ep):
        logits, labels = ep
        probs = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
        preds = (probs >= 0.5).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(labels, preds,
                                                        average='binary', zero_division=0)
        return {'accuracy': accuracy_score(labels, preds), 'precision': pr,
                'recall': rc, 'f1': f1, 'roc_auc': roc_auc_score(labels, probs)}

    common = dict(output_dir='/content/ckpt2', num_train_epochs=3, learning_rate=2e-5,
                  warmup_ratio=0.1, weight_decay=0.01, per_device_train_batch_size=BATCH,
                  gradient_accumulation_steps=GRAD_ACCUM, per_device_eval_batch_size=64,
                  fp16=torch.cuda.is_available(), seed=SEED, load_best_model_at_end=True,
                  metric_for_best_model='f1', save_total_limit=1, logging_steps=200,
                  report_to='none')
    try:
        targs = TrainingArguments(eval_strategy='epoch', save_strategy='epoch', **common)
    except TypeError:
        targs = TrainingArguments(evaluation_strategy='epoch', save_strategy='epoch', **common)

    trainer = Trainer(model=model, args=targs, train_dataset=ds['train'],
                      eval_dataset=ds['val'], data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    trainer.train()
    trainer.save_model(BEST_DIR); tok.save_pretrained(BEST_DIR)
    json.dump({'base_model': used_model, 'max_len': MAX_LEN, 'seed': SEED,
               'run': RUN_TAG}, open(f'{BEST_DIR}/train_info.json', 'w'), indent=2)
    print('Phase 3 complete ->', BEST_DIR)


## Phase 4 — Test metrics: overall + per source
Loads the best checkpoint fresh. Reports overall test metrics **and a breakdown by
source** (PhiUSIIL-domain test URLs vs malicious_phish-domain test URLs). RUN 1's
failure showed AUC 0.53 on the other source; if both per-source AUCs here are high,
the shortcut is gone.

In [ ]:
# ============ Phase 4 — Evaluate (overall + per source) ============
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BEST_DIR = f'{BASE}/best_model'
METRICS_PATH = f'{BASE}/eval/metrics_test.json'
tok = AutoTokenizer.from_pretrained(BEST_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).eval().to(DEVICE)

def get_logits(urls, bs=256):
    out = []
    with torch.no_grad():
        for i in range(0, len(urls), bs):
            enc = tok(list(urls[i:i+bs]), truncation=True, max_length=128,
                      padding=True, return_tensors='pt').to(DEVICE)
            out.append(model(**enc).logits.float().cpu())
    return torch.cat(out).numpy()

def scores(y, p):
    preds = (p >= 0.5).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, preds, average='binary',
                                                    zero_division=0)
    return {'n': int(len(y)), 'accuracy': float(accuracy_score(y, preds)),
            'precision': float(pr), 'recall': float(rc), 'f1': float(f1),
            'roc_auc': float(roc_auc_score(y, p)) if len(set(y)) > 1 else None}

if os.path.exists(METRICS_PATH):
    print('Phase 4 already completed -- skipping.')
    print(json.dumps(json.load(open(METRICS_PATH)), indent=2))
else:
    test = splits['test']
    logits = get_logits(test['url'].tolist())
    probs = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    labels = test['label'].to_numpy()
    np.save(f'{BASE}/eval/test_logits.npy', logits)

    result = {'overall': scores(labels, probs), 'per_source': {}}
    for src in sorted(test['source'].unique()):
        m = (test['source'] == src).to_numpy()
        result['per_source'][src] = scores(labels[m], probs[m])
    print(json.dumps(result, indent=2))
    json.dump(result, open(METRICS_PATH, 'w'), indent=2)

    cm = confusion_matrix(labels, (probs >= 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=['legit', 'phish']).plot(
        cmap='Blues', colorbar=False)
    plt.title('Method 1 RUN 2 — Test Confusion Matrix')
    plt.savefig(f'{BASE}/eval/confusion_matrix.png', dpi=160, bbox_inches='tight'); plt.close()
    fpr, tpr, _ = roc_curve(labels, probs)
    plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], '--', c='gray')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(f"ROC AUC={result['overall']['roc_auc']:.4f}")
    plt.savefig(f'{BASE}/eval/roc_curve.png', dpi=160, bbox_inches='tight'); plt.close()

    err = pd.DataFrame({'url': test['url'], 'label': labels, 'p_url': probs})
    err[(err.label == 1) & (err.p_url < 0.5)].nsmallest(10, 'p_url').to_csv(
        f'{BASE}/eval/false_negatives.csv', index=False)
    err[(err.label == 0) & (err.p_url >= 0.5)].nlargest(10, 'p_url').to_csv(
        f'{BASE}/eval/false_positives.csv', index=False)
    print('Phase 4 complete.')


## Phase 5 — Temperature calibration
Fits T on the validation split (LBFGS/NLL); reports ECE on test before/after.
Ships as `temperature.json`; the backend applies `softmax(logits / T)`.

In [ ]:
# ============ Phase 5 — Temperature scaling + ECE ============
TEMP_PATH = f'{BASE}/artifacts/temperature.json'

def ece(p, y, n=10):
    bins = np.linspace(0, 1, n + 1); tot = 0.0
    conf = np.maximum(p, 1 - p); correct = ((p >= 0.5).astype(int) == y)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum(): tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(tot)

if os.path.exists(TEMP_PATH):
    print('Phase 5 already completed -- skipping.', json.load(open(TEMP_PATH)))
else:
    val = splits['val']
    vl = torch.tensor(get_logits(val['url'].tolist()))
    vy = torch.tensor(val['label'].to_numpy())
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.05, max_iter=200)
    nll = torch.nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad(); loss = nll(vl / T.clamp(min=1e-3), vy); loss.backward(); return loss
    opt.step(closure)
    temperature = float(T.detach().clamp(min=1e-3))
    tl = torch.tensor(np.load(f'{BASE}/eval/test_logits.npy'))
    ty = splits['test']['label'].to_numpy()
    pb = torch.softmax(tl, -1)[:, 1].numpy()
    pa = torch.softmax(tl / temperature, -1)[:, 1].numpy()
    eb, ea = ece(pb, ty), ece(pa, ty)
    print(f'T={temperature:.4f}  ECE {eb:.4f} -> {ea:.4f}')
    json.dump({'temperature': temperature, 'ece_before': eb, 'ece_after': ea},
              open(TEMP_PATH, 'w'), indent=2)
    print('Phase 5 complete.')


## Phase 6 — Real-world sanity set (the direct fix check)
A small hand-labelled set of well-known URLs. RUN 1 failed this obviously
(google.com/maps scored 1.000). RUN 2 should score benign well-known sites LOW and
clear phishing patterns HIGH. This is the concrete, demo-ready evidence for the
report that the model now behaves sensibly on real inputs.

In [ ]:
# ============ Phase 6 — Real-world sanity set ============
SANITY = [
    ('https://www.google.com/maps', 0), ('https://www.youtube.com/watch?v=x', 0),
    ('https://github.com/pytorch/pytorch', 0), ('https://en.wikipedia.org/wiki/QR_code', 0),
    ('https://www.maybank2u.com.my/', 0), ('https://www.imda.gov.sg/', 0),
    ('https://www.utar.edu.my/', 0), ('https://outlook.office.com/mail/', 0),
    ('https://www.paypal.com/signin', 0), ('https://shopee.com.my/', 0),
    ('http://paypal-secure-verify.top/login/update.php', 1),
    ('http://maybank2u-verify.xyz/login', 1),
    ('http://203.0.113.7/account/confirm', 1),
    ('http://apple.id-login.secure-check.tk/', 1),
    ('http://update-account.cimb-my.com.security-alert.ga/', 1),
    ('http://free-gift-card.win/claim?id=123', 1),
    ('http://bankofamerica.com.login-verify.ru/', 1),
    ('http://192.168.0.1.phishing-site.xyz/reset', 1),
    ('http://microsoft-support-alert.ml/warning', 1),
    ('http://verify-shopee-wallet.cf/pay', 1),
]
temperature = json.load(open(f'{BASE}/artifacts/temperature.json'))['temperature']
urls = [u for u, _ in SANITY]; y = np.array([l for _, l in SANITY])
logits = get_logits(urls)
p = torch.softmax(torch.tensor(logits) / temperature, -1)[:, 1].numpy()
acc = float(((p >= 0.5).astype(int) == y).mean())
print(f'Sanity accuracy: {acc:.2f}  (RUN 1 would fail the benign half)\n')
for (u, lab), pr in sorted(zip(SANITY, p), key=lambda z: z[1]):
    ok = '  ' if (pr >= 0.5) == bool(lab) else '!!'
    print(f'{ok} p_url={pr:.3f}  [{"phish" if lab else "benign"}]  {u}')
json.dump({'sanity_accuracy': acc,
           'results': [{'url': u, 'label': l, 'p_url': float(pr)}
                       for (u, l), pr in zip(SANITY, p)]},
          open(f'{BASE}/eval/sanity_set.json', 'w'), indent=2)
print('\nPhase 6 complete ->', f'{BASE}/eval/sanity_set.json')


## Phase 7 — ONNX + INT8 + latency + `predict_url()`
Same deployment pipeline as RUN 1 (validated: 0 pp INT8 drop, 37 ms/URL). Exports,
quantizes, checks the ≤2 pp policy, benchmarks CPU latency, defines `predict_url`.
Download `run2_combined/artifacts/` into the repo `training/artifacts/`.

In [ ]:
# ============ Phase 7 — ONNX + INT8 + latency + predict_url ============
ART = f'{BASE}/artifacts'
from optimum.onnxruntime import ORTModelForSequenceClassification
import onnxruntime as ort_rt
from onnxruntime.quantization import quantize_dynamic, QuantType

if not os.path.exists(f'{ART}/onnx_fp32/model.onnx'):
    ORTModelForSequenceClassification.from_pretrained(BEST_DIR, export=True
        ).save_pretrained(f'{ART}/onnx_fp32')
    print('Exported ONNX FP32.')
else:
    print('ONNX FP32 exists -- skipping.')
QUANT = f'{ART}/model_quant.onnx'
if not os.path.exists(QUANT):
    quantize_dynamic(f'{ART}/onnx_fp32/model.onnx', QUANT, weight_type=QuantType.QInt8)
    print('Quantized INT8.')
tok.save_pretrained(ART)

def sess(path, threads=1):
    so = ort_rt.SessionOptions(); so.intra_op_num_threads = threads
    return ort_rt.InferenceSession(path, so, providers=['CPUExecutionProvider'])
def onnx_logits(s, urls, bs=64):
    names = {i.name for i in s.get_inputs()}; out = []
    for i in range(0, len(urls), bs):
        enc = tok(list(urls[i:i+bs]), truncation=True, max_length=128,
                  padding=True, return_tensors='np')
        out.append(s.run(None, {k: v.astype(np.int64) for k, v in enc.items()
                                if k in names})[0])
    return np.concatenate(out)

samp = splits['test'].sample(n=min(2000, len(splits['test'])), random_state=SEED)
y = samp['label'].to_numpy()
def f1_of(lg): return precision_recall_fscore_support(
    y, (torch.softmax(torch.tensor(lg), -1)[:, 1].numpy() >= 0.5).astype(int),
    average='binary', zero_division=0)[2]
f1_torch = f1_of(get_logits(samp['url'].tolist()))
s_fp32, s_int8 = sess(f'{ART}/onnx_fp32/model.onnx'), sess(QUANT)
f1_fp32 = f1_of(onnx_logits(s_fp32, samp['url'].tolist()))
f1_int8 = f1_of(onnx_logits(s_int8, samp['url'].tolist()))
drop = (f1_torch - f1_int8) * 100
print(f'F1 torch {f1_torch:.4f} | fp32 {f1_fp32:.4f} | int8 {f1_int8:.4f} | drop {drop:.2f} pp')
DEPLOY = QUANT if drop <= 2.0 else f'{ART}/onnx_fp32/model.onnx'
if drop > 2.0: print('INT8 drop > 2 pp -> deploying FP32.')
json.dump({'deploy_model': os.path.basename(DEPLOY)}, open(f'{ART}/deploy_choice.json', 'w'))

model_cpu = model.to('cpu').eval()
bu = 'https://login.secure-account-verify.example.com/session/check?id=12345'
def bench(fn, runs=200, warm=20):
    for _ in range(warm): fn()
    t = [ (lambda t0: (fn(), time.perf_counter()-t0)[1])(time.perf_counter()) for _ in range(runs)]
    a = np.array(t)*1000; return float(np.median(a)), float(np.percentile(a, 95))
enc = tok(bu, truncation=True, max_length=128, return_tensors='pt')
lat = {}
with torch.no_grad(): lat['pytorch_cpu'] = bench(lambda: model_cpu(**enc))
lat['onnx_fp32'] = bench(lambda: onnx_logits(s_fp32, [bu]))
lat['onnx_int8'] = bench(lambda: onnx_logits(s_int8, [bu]))
print('\nLatency median/P95 ms:', {k: (round(m,1), round(p,1)) for k,(m,p) in lat.items()})
model.to(DEVICE)

_T = json.load(open(f'{ART}/temperature.json'))['temperature']
_ds = sess(DEPLOY)
def predict_url(url: str) -> float:
    return float(torch.softmax(torch.tensor(onnx_logits(_ds, [url])[0]) / _T, -1)[1])
print('\npredict_url() demo:')
for u in ['https://www.google.com/maps', 'http://paypal-secure-verify.top/login/update.php',
          'https://bit.ly/3xYzAb', 'https://login.maybank2u.com.my/',
          'http://203.0.113.7/account/confirm']:
    print(f'  p_url={predict_url(u):.3f}  {u}')

summary = {'run': RUN_TAG, 'test_metrics': json.load(open(METRICS_PATH)),
           'calibration': json.load(open(f'{ART}/temperature.json')),
           'sanity': json.load(open(f'{BASE}/eval/sanity_set.json')),
           'quantization': {'f1_torch': float(f1_torch), 'f1_onnx_int8': float(f1_int8),
                            'drop_pp': float(drop), 'deployed': os.path.basename(DEPLOY)},
           'latency_ms_median_p95': lat}
json.dump(summary, open(f'{ART}/metrics_summary.json', 'w'), indent=2)
print('\nPhase 7 complete. Download', ART, '-> QRGuard/training/artifacts/')


## Done — RUN 2 checklist
Download `MyDrive/FYP2/method1/run2_combined/artifacts/` into
`QRGuard/training/artifacts/`. **Judge success by:**
1. Phase 6 sanity accuracy high AND google.com/maps scores LOW.
2. Phase 4 per-source ROC-AUC high for BOTH sources (RUN 1 cross-source was 0.53).
3. Phase 7 `predict_url` demo: benign low, phishing high.

Record the RUN 2 measurements and conclusions for the FYP report.